Notebook to test our model on real data. This is a kind of test set.  


I start with the most obvious sample categories:
- clusters of trash
- Scattered litter
- Less litter, scattered
- hawkers
- flowers
- rocks

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import cv2
import random
from mtrain.utils import show, DiskImage, DiskBooleanMask, overlay_mask_on_img as OV
from mtrain.example_dir import ExampleDir, load_npz
from mtrain.example_dir.iterdir import get_dirs
from mtrain.example_dir.defaults import default_negmask_learners, default_smallnet_learners
from tqdm import tqdm
from mtrain.example_dir.learners import step_downer
from mtrain.neg_mask.model.predict import trash as pred_trash
from mtrain.denorm import denormalize_imagenet

In [ ]:
negmask = default_negmask_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "latest", "high-recall"], 4)

In [ ]:
negmask["latest"]

In [ ]:
nml = negmask["md"]
print(nml)
base = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set/299603061792440")
image = DiskImage.load(base / "image.jpg")
mask = DiskBooleanMask.load(base / "m2-md.png")



ds, _ = pred_trash.get_model_inputs(
    image,
    mask,
    nml.crop_size,
    nml.bbox_pad,
    nml.mutator,
    valid_tfms_crop_size=nml.valid_tfms_crop_size,
)
plt.imshow(denormalize_imagenet(ds[9]).permute([1,2,0]))
plt.show()

In [ ]:
smallnet = default_smallnet_learners(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models"), ["md", "sm"], 4)

In [ ]:
DS = Path("/Users/hariomnarang/Desktop/personal/roads/datasets")
DELHI_SAMPLES_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test"
)
TEST_SET_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set"
)
DAKOR_IMAGES = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/images")
DAKOR_TEST_SET_PATH = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/dakor_test_set"
)

SINGLE_TEST_PATH = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/testing/single")
BATCH_TEST_PATH = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/testing/batch")

# Test batch prediction with normal pred

In [ ]:

from tqdm import tqdm

sdirs = sorted(get_dirs(SINGLE_TEST_PATH))[:8]
bdirs = sorted(get_dirs(BATCH_TEST_PATH))[:8]

In [ ]:
sdirs

In [ ]:
bdirs

In [ ]:

sedirs = [ExampleDir(d, smallnet, negmask) for d in sdirs]
bedirs = [ExampleDir(d, smallnet, negmask) for d in bdirs]



In [ ]:
for ed in tqdm(sedirs):
    ed.smallnet_mask_path("md")

In [ ]:
ExampleDir.batch_predict_smallnet_masks(smallnet["md"], bedirs, 4)

In [ ]:
for (s,b) in zip(sdirs, bdirs):
    smask = DiskBooleanMask.load(s / "mask-md.png")
    bmask = DiskBooleanMask.load(b / "mask-md.png")

    np.all(smask == bmask)

In [ ]:
for ed in tqdm(sedirs):
    ed.negmask_paths("latest", "md")

In [ ]:
ExampleDir.batch_predict_negmask_masks(negmask["latest"], bedirs, "md")

In [ ]:
from mtrain.example_dir.core import load_npz
for (s,b) in zip(sdirs, bdirs):
    oname, tname = negmask["latest"].pathnames
    np.all(load_npz(s / oname) == load_npz(b / oname))
    np.all(load_npz(s / tname) == load_npz(b / tname))
    smask = DiskBooleanMask.load(s / "mask-md.png")
    bmask = DiskBooleanMask.load(b / "mask-md.png")

    np.all(smask == bmask)

In [ ]:
from mtrain.utils import overlay_mask_on_img as OV
res = []
for b in bdirs:
    oname, tname = negmask["latest"].pathnames
    other = load_npz(b / oname)
    trash = load_npz(b / tname)
    is_trash = trash > other
    bmask = DiskBooleanMask.load(b / "m2-md.png")
    image = DiskImage.load(b / "image.jpg")

    res.append((image, OV(image, bmask & is_trash)))


In [ ]:
from mtrain.utils import *

show(it_chain(res[:4]))

# Analyze test set

In [ ]:
from tqdm import tqdm
ds = list(get_dirs(DAKOR_TEST_SET_PATH)) + list(get_dirs(TEST_SET_PATH))

In [ ]:


print("evaluating")
res = []
for d in tqdm(ds):
    edir = ExampleDir(d, smallnet, negmask)

    md_smp = DiskBooleanMask.load(edir.trimmed_mask_path("md",False))

    image = DiskImage.load(edir.image_path)

    edir.negmask_paths("latest", "md", force=True)
    edir.negmask_paths("md", "md", force=True)

    res.append((image, md_smp, edir.get_trash_mask("latest", "md"), edir.get_trash_mask("md", "md")))

In [ ]:
from mtrain.utils import overlay_mask_on_img as OV, it_chain

res2 = [(i, OV(i, md),  OV(i, mdtmask), OV(i, recall_tmask)) for (i,md,mdtmask,recall_tmask) in res]

len(res2)


In [ ]:
edir = ExampleDir(ds[11], smallnet, negmask)
image, mask = DiskImage.load(edir.image_path), DiskBooleanMask.load(edir.trimmed_mask_path("md"))
show([image, mask])

In [ ]:
nml = negmask["high-recall"]
ot, tt = pred_trash.predict_and_return_prob_masks_using_unblurred(
    image, mask, nml.learner, nml.crop_size, None, 5, nml.mutator, nml.valid_tfms_crop_size
)
tmask = (tt > ot)

In [ ]:
show([image, mask, tmask], ncols=3)

In [ ]:
from mtrain.neg_mask.crops import get_region_crops, padded_bbox

bboxes = list(get_region_crops(mask))
bboxes

In [ ]:
bb = bboxes[9]
showimg = image.copy()
bb = padded_bbox(bb, 20, mask.shape)
cv2.rectangle(showimg, (bb.x,bb.y), (bb.x2,bb.y2), [255,0,0], 3)
show([showimg], ncols=1)

In [ ]:
crops, masks, bboxes, inner_bboxes = pred_trash.get_crops_masks_bboxes(
    image, mask, nml.crop_size, nml.bbox_pad
)

In [ ]:
bboxes


In [ ]:
ipds, _ = pred_trash.get_model_inputs(image, mask, nml.crop_size, nml.bbox_pad, nml.mutator, nml.valid_tfms_crop_size)

In [ ]:
show([image, denormalize_imagenet(ipds[4]).permute([1,2,0])])

In [ ]:
show([image,mask, tmask], ncols=3)

In [ ]:
show(it_chain(res2[10:15]), (25,40), 4, cmap="gray", axis="off")

In [ ]:
plt.imshow(plt.imread(ds[6] / "image.jpg"))

In [ ]:
res[6][0]

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox
from mtrain.example_dir.learners import step_downer
from mtrain.neg_mask.model.predict import trash as pred_trash

def noise_overwriter(cropped_image, mask, inner_bbox):
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.overwrite_with_noise(20)
    return tfm.crop

image = DiskImage.load(ds[6] / "image.jpg")
m2 = DiskBooleanMask.load(ds[6] / "m2.png")
blur_ds, bboxes = pred_trash.get_model_inputs(image, m2, 224, 10, noise_overwriter)

show([image, m2])

In [ ]:
from mtrain.denorm import denormalize_imagenet
i = 26
plt.imshow(denormalize_imagenet(blur_ds[i]).permute([1,2,0]).numpy())

In [ ]:
import torch
with torch.no_grad():
    res = negmask["latest"].learner.model(blur_ds[i].unsqueeze(0))
    print(res)
    

In [ ]:
from mtrain.neg_mask.model.gradcam import show_gradcam_for_image
show(show_gradcam_for_image(negmask["latest"].learner, blur_ds[i].unsqueeze(0), 1, "0.7.1.convpath.1.0"))

In [ ]:
d = ds[0]
edir = ExampleDir(d, smallnet, negmask)
smp = edir.smallnet_mask_path("md", True)
op, tp = edir.negmask_paths("md", "md", True)
image = DiskImage.load(edir.image_path)
smp = DiskBooleanMask.load(smp)
op = load_npz(op)
tp = load_npz(tp)
m = tp > op
show([
    OV(image, smp), OV(image, m)
])

In [ ]:
LOTS_OF_LITTER = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/532628267735038",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/271937142353543",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1407390010751492",
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/samples_mapillary/100/26210846388517270"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/933090527327250"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/904411971827702"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/409495874297166"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/471611377923092"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/1132072153938604"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/trash/delhi_litter/318666179638135"
    ),
]

BROKEN_ROAD = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1765386860738037",
]

SCATTERED = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/167994091824213",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/671811303998480",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/173239464669959",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/125035310018047",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1634936550227999",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/291967965907673",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1384373959095157",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/196244582323540",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/801754743801999",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/822560908385930",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/299603061792440",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1397649970633008",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2985603351766082",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/144348374637763",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/843858569843204",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/332198098541859",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/3625083887603287",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2769241939938889",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/669848760702786",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/955227375338093",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/212281790352108",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/214125100125056",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1490763574593712",
]

BLUR = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/161598622858285",
]

FLOWERS = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2800723576860190",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/871543466734475",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2562170430756718",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/2926084337660344",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/1043236500247070",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/930842434423691",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/550822005934553",
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/935180533980655"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1019440569393347"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1217095282865379"
    ),
    Path(
        "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/857578969238303"
    ),
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1253130721794975",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/831014837763053",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/322214212632654",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/3799706853575740",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/369602612275380",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1000795241832418",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1357433225213341",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/test-samples/neg-masking/V1/rocks/classification/flowers/1131950611202217",
]


STONES = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/566935955515605",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/195067892836607",
]

WALLS = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/884772330133126",
]

WEIRD = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/8792094944191192",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/810183961580401",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/672731224796321",
]

WIDE_ANGLE = [
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/992742956405587",
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/535408762984577",
]

ALL_DIRS = (
    WIDE_ANGLE
    + WEIRD
    + WALLS
    + STONES
    + FLOWERS
    + BLUR
    + SCATTERED
    + BROKEN_ROAD
    + LOTS_OF_LITTER
)

# add samples to test set

In [ ]:
def get_image_dirs(samples_dir):
    for d in samples_dir.glob("*"):
        if not d.is_dir() or not (d / "image.jpg").exists():
            continue
        yield d


it = get_image_dirs(DELHI_SAMPLES_DIR)


In [ ]:
p = next(it)
print(p)
show([plt.imread(p / "image.jpg")], ncols=1)

## move samples to test set

In [ ]:
from mtrain.example_dir import create_dirs_for_images, ExampleDir, run_bulk_inference

In [ ]:
import shutil

for d in ALL_DIRS:
    dest = TEST_SET_PATH / Path(d).name
    if not dest.exists():
        shutil.copytree(d, dest)

# run bulk inference

In [ ]:
# force run

dirs = [d for d in TEST_SET_PATH.glob("*") if d.is_dir()]
DATA_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/128x128-all-data"
)
MODELS_DIR = Path("/Users/hariomnarang/Desktop/personal/roads/datasets/models/")


smallnet_128x128_learner = get_smallnet_learner(
    128,
    4,
    DATA_DIR,
    MODELS_DIR / "smallnet-128x128" / "xresnet18-iter17.pth",
    arch="xresnet18",
    device=default_device(),
)
smallnet_64x64_learner = get_smallnet_learner(
    64,
    4,
    DATA_DIR,
    MODELS_DIR / "smallnet-64x64" / "raw_torch_iter80.pth",
    arch="xresnet18",
    device=default_device(),
)
NEGMASK_224_STEP_EDGE_NEW = "/Users/hariomnarang/Desktop/personal/roads/datasets/models//"
negmask_224x224_learner = get_negmask_learner(4, 224, MODELS_DIR / "successive-224" / "st_ed_tfm0-final-all-data-with-taco-iter-25.pth")


for d in tqdm(dirs):
    edir = ExampleDir(
        d, smallnet_128x128_learner, smallnet_64x64_learner, negmask_224x224_learner
    )
    edir.trash_probs_path(True)
    edir.other_probs_path(True)
    # edir.trash_50x50_probs_path(True)
    # edir.other_50x50_probs_path(True)
    # edir.flower_neg_probs_path()
    # edir.flower_pos_probs_path()
    # edir.smallnet_50x50_path(force=True)

# Visualize results

In [ ]:
dirs = [d for d in TEST_SET_PATH.glob("*") if d.is_dir()]
len(dirs)

In [ ]:
def get_areas(binary_mask):
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(
        binary_mask, connectivity=8
    )

    areas = []
    lengths = []
    for label in range(1, num_labels):  # skip label 0 (background)
        areas.append(stats[label, cv2.CC_STAT_AREA])
        lengths.append(stats[label, cv2.CC_STAT_HEIGHT])

    return areas, lengths

In [ ]:
idx = 0

In [ ]:
dirs[idx]

In [ ]:
from mtrain.example_dir.core import loaded_mask_name_by_masks, load_npz
# from mtrain.smallnet.unet.predict.strided import single
# from mtrain.example_dir.core import (
#     NEGMASK_BLUR_KERNEL_SZ,
#     NEGMASK_BLUR_KERNEL_SIGMA,
#     NEGMASK_BBOX_PAD,
# )
# from mtrain.utils import draw_grid_cv2

edir = ExampleDir(dirs[idx])
# edir = ExampleDir(Path("/Users/hariomnarang/Desktop/personal/roads/datasets/inference/delhi_sample_500_results_test/167994091824213"))

image = DiskImage.load(edir.image_path)
mask100x100 = DiskBooleanMask.load(edir.trimmed_mask_path())
# mask50x50 = DiskBooleanMask.load(edir.trimmed_50x50_mask_path())

mask_by_path_100x100 = edir.negmask_100x100_paths()
edir.generate_negmask_probs()
trash, other = load_npz(edir.trash_probs_path()), load_npz(edir.other_probs_path())

# mask_by_path_50x50 = edir.negmask_50x50_paths()
# mask_100x100 = loaded_mask_name_by_masks(mask_by_path_100x100)
# mask_50x50 = mask_100x100

# def _get_bin_masks(m1, m2):
#     return (m1["trash"] > m1["other"]) | (m2["trash"] > m2["other"])

# print(mask_by_path_100x100)
# print(mask_by_path_50x50)
show(
    [
        OV(image, mask100x100),
        OV(image, trash > other),
        # OV(image, _get_bin_masks(mask_100x100["step_edge"], mask_50x50["step_edge"])),
        # OV(image, _get_bin_masks(mask_100x100["step_edge_224"], mask_50x50["step_edge_224"])),
        # OV(image, _get_bin_masks(mask_100x100["unblurred"], mask_50x50["unblurred"])),
    ],
    (20, 20),
    2,
)

idx += 1

In [ ]:
se = _get_bin_masks(mask_100x100["step_edge"], mask_50x50["step_edge"])
se224 = _get_bin_masks(mask_100x100["step_edge_224"], mask_50x50["step_edge_224"])
unb = _get_bin_masks(mask_100x100["unblurred"], mask_50x50["unblurred"])

In [ ]:
show(
    [
        se,
        se224,
        unb,
        se & ~se224,
        se & ~unb,
        se224 & ~se,
        se224 & ~unb,
        unb & ~se,
        unb & ~se224,
    ],
    (20, 20),
    3,
)


In [ ]:
direc = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set/125035310018047"
)
direc = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/inference/test_set/125035310018047"
)


edir = ExampleDir(direc)
plt.imshow(plt.imread(edir.image_path))

In [ ]:
from functools import partial
from mtrain.neg_mask.model.predict.trash import step_down_edge_tfm
from mtrain.neg_mask.model.datasets.blur_pad_dl import (
    BlurPadInferDataset,
    CropTfmsOutsideBbox,
)
from mtrain.neg_mask.crops import get_region_crops, padded_bbox
from mtrain.neg_mask.crops import padded_crop, bbox_only_mask, Bbox

mask = DiskBooleanMask.load(edir.trimmed_mask_path())
image = DiskImage.load(edir.image_path)
bboxes = list(get_region_crops(mask))
bboxes = [padded_bbox(bbox, 10, mask.shape) for bbox in bboxes]

crops, masks = [], []
inner_bboxes = []
for bbox in bboxes:
    tight_img, new_y1, new_x1 = padded_crop(image, bbox, 128)
    tight_mask = bbox_only_mask(mask, bbox, 128)
    inner_bbox = Bbox(bbox.x - new_x1, bbox.y - new_y1, bbox.w, bbox.h)
    crops.append(tight_img)
    masks.append(tight_mask)
    inner_bboxes.append(inner_bbox)


def step_down_gauss_tfm(cropped_image, mask, inner_bbox):
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(0.3)
    return tfm.crop


ds = BlurPadInferDataset(
    crops,
    masks,
    inner_bboxes,
    128,
    partial(step_down_edge_tfm, ratio=0.5),
)

# crops, masks, result_bboxes = [], [], []
# for bbox in tqdm(bboxes):
#     inner_bbox = Bbox(bbox.x - new_x1, bbox.y - new_y1, bbox.w, bbox.h)
#     crops.append(tight_img)
#     masks.append(tight_mask)
#     result_bboxes.append(inner_bbox)

In [ ]:
i -= 1

In [ ]:
# from mtrain.example_dir.core import NEG_MASK_STEP_EDGE_MODEL_PATH
from mtrain.example_dir.core import NEGMASK_224_STEP_EDGE

from mtrain.example_dir import get_default_negmask_learner
from mtrain.denorm import denormalize_imagenet

tens = ds[i]
learner = get_default_negmask_learner(NEGMASK_224_STEP_EDGE)
probs = learner.model(tens.unsqueeze(0)).softmax(dim=1)
img = denormalize_imagenet(tens).permute([1, 2, 0]).numpy()
print(probs)
plt.imshow(img)
plt.show()
i += 1

In [ ]:
from mtrain.neg_mask.model.gradcam import show_gradcam_for_image

show(show_gradcam_for_image(learner, tens.unsqueeze(0), 1, "0.7.1.convpath.1.0"))

In [ ]:
# bbox = bboxes[i]
# tight_img, new_y1, new_x1 = padded_crop(image, bbox, 128)
# tight_mask = bbox_only_mask(mask, bbox, 128)
# inner_bbox = Bbox(bbox.x - new_x1, bbox.y - new_y1, bbox.w, bbox.h)

# print(inner_bbox)
# show([tight_img, tight_mask])

# compare models

In [ ]:
from mtrain.neg_mask.model.predict.trash import (
    predict_and_return_prob_masks_using_unblurred,
)

In [ ]:
from mtrain.example_dir.core import dummy_unblur_dls
from fastai.callback.all import ProgressCallback
import torch
from torchvision.models import resnet18
from fastai.vision.all import vision_learner, ImageDataLoaders
from fastai.basics import load_learner, F1Score, CrossEntropyLossFlat, get_image_files

# NEGMASK_UNBLURRED_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-p10-iter100-verified.pt"

# unblur_stage1_learner = vision_learner(
#     dummy_unblur_dls(),
#     resnet18,
#     metrics=[F1Score(average="macro")],
#     loss_func=CrossEntropyLossFlat(weight=torch.tensor([1.41, 3.43])),
#     pretrained=True,
#     n_out=2,
# )
# unblur_stage1_learner = unblur_stage1_learner.remove_cb(ProgressCallback)
# sd = torch.load(
#     "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/resnet18-stage1-iter15.pth"
# )
# unblur_stage1_learner.model.load_state_dict(sd)

In [ ]:
from mtrain.neg_mask.model.datasets.blur_pad_dl import CropTfmsOutsideBbox


def step_down_gauss_tfm(cropped_image, mask, inner_bbox):
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down_gaussian(0.3)
    return tfm.crop


In [ ]:
from mtrain.example_dir.core import get_default_negmask_learner

# NEGMASK_UNBLURRED_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/unblurred/v2/blurpad/fullset-p10-iter100-verified.pt"
# NEGMASK_STEP_DOWN_GAUSSIAN_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur/tfm-gaussstepdown_min-3_samples-all-xresnet18_iter-50.pth"
# NEG_MASK_STEP_EDGE_MODEL_PATH = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-unblur/tfm-stepdown_ratio-5_samples-all-xresnet18_iter-20.pth"
# NEGMASK_BASELINE = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/neg-baseline/baseline-pad-10-crop-size-64-iter-20.pt"
NEGMASK_224_STEP_EDGE = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-iter-20.pth"
NEGMASK_224_STEP_EDGE_NEW = "/Users/hariomnarang/Desktop/personal/roads/datasets/models/successive-224/st_ed_tfm0-final-all-data-with-taco-iter-25.pth"

# unblurred_v1_learner = get_default_negmask_learner(NEGMASK_UNBLURRED_MODEL_PATH, "resnet18")
# step_down_gauss_learner = get_default_negmask_learner(
#     NEGMASK_STEP_DOWN_GAUSSIAN_MODEL_PATH, "xresnet18"
# )
# step_edge_learner = get_default_negmask_learner(
#     NEG_MASK_STEP_EDGE_MODEL_PATH, "xresnet18"
# )
step_edge_224_learner = get_default_negmask_learner(NEGMASK_224_STEP_EDGE, "xresnet18")
new_step_edge_224_learner = get_default_negmask_learner(
    NEGMASK_224_STEP_EDGE_NEW, "xresnet18"
)

In [ ]:
i = 0

In [ ]:
# from mtrain.neg_mask.model.datasets.blur_pad_dl import blur_overwriter
from functools import partial


def step_down_edge_tfm(cropped_image, mask, inner_bbox, ratio):
    tfm = CropTfmsOutsideBbox(cropped_image, inner_bbox)
    tfm = tfm.step_down(ratio)
    return tfm.crop


edir = ExampleDir(dirs[i])
image, mask = (
    DiskImage.load(edir.image_path),
    DiskBooleanMask.load(edir.trimmed_mask_path()),
)
new_step_224_other, new_step_224_trash = predict_and_return_prob_masks_using_unblurred(
    image,
    mask,
    new_step_edge_224_learner,
    224,
    bbox_pad=10,
    mutator=partial(step_down_edge_tfm, ratio=0.5),
)

step_224_other, step_224_trash = predict_and_return_prob_masks_using_unblurred(
    image,
    mask,
    step_edge_224_learner,
    224,
    bbox_pad=10,
    mutator=partial(step_down_edge_tfm, ratio=0.5),
)

show(
    [
        # OV(image, unblur_trash > unblur_other),
        OV(image, mask),
        # OV(image, step_trash > step_other),
        OV(image, new_step_224_trash > new_step_224_other),
        OV(image, step_224_trash > step_224_other),
    ],
    (30, 30),
    3,
    "off",
)

# show([image, mask, trash > other, OV(image, trash > other)], (20,20), 2, "off")

In [ ]:
i += 1

# compare smallnet

In [ ]:
from fastai.vision.all import *

DATA_DIR = Path(
    "/Users/hariomnarang/Desktop/personal/roads/datasets/T007-uncentered/128x128-all-data"
)
new_smallnet_path = "/Users/hariomnarang/Desktop/gdrive-sync/garbage/experiments/128x128-bright-objects-xresnet18-01/log/xresnet18-iter10"
dls = SegmentationDataLoaders.from_label_func(
    ".",
    bs=4,
    fnames=get_image_files(DATA_DIR / "images"),
    label_func=lambda o: DATA_DIR / "masks" / f"{o.stem}.png",
    codes=np.array(["background", "trash"]),
    item_tfms=Resize(128),
    num_workers=8,
    # pin_memory=True,
    persistent_workers=True,
    batch_tfms=aug_transforms(),
)

In [ ]:
class CombinedLoss:
    "Dice and Focal combined"

    def __init__(
        self, axis=1, smooth=1.0, alpha=1.0, weights=torch.tensor([1.0, 2.0]).to("mps")
    ):
        store_attr()
        self.focal_loss = FocalLossFlat(axis=axis, weight=weights)
        self.dice_loss = DiceLoss(axis, smooth)

    def __call__(self, pred, targ):
        return self.focal_loss(pred, targ) + self.alpha * self.dice_loss(pred, targ)

    def decodes(self, x):
        return x.argmax(dim=self.axis)

    def activation(self, x):
        return F.softmax(x, dim=self.axis)


learner = unet_learner(
    dls, xresnet18, n_out=2, pretrained=True, loss_func=CombinedLoss(), metrics=[Dice()]
)
learner.model.load_state_dict(torch.load(new_smallnet_path, "mps"))
learner = learner.remove_cb(ProgressCallback)


In [ ]:
learner.show_results()

In [ ]:
from mtrain.example_dir.iterdir import get_dirs
from mtrain.smallnet.unet.predict.strided import single
from mtrain.utils import *

dirs = list(get_dirs(TEST_SET_PATH))

In [ ]:
d = dirs[18]
edir = ExampleDir(d)

img_arr = edir.load_and_resize_image()
mask = single.strided_predict_unet_only_mask(img_arr, 128, learner, [64])

elev_pred = DiskBooleanMask.load(edir.elev_mask_path())
mapi_pred = DiskBooleanMask.load(edir.mapi_mask_path())
new_m2 = edir._get_trimmed_mask(mask, elev_pred, mapi_pred)
m2_50x50 = DiskBooleanMask.load(edir.trimmed_50x50_mask_path())
orig_m2 = DiskBooleanMask.load(edir.trimmed_mask_path())

show(
    [
        new_m2,
        overlay_mask_on_img(img_arr, new_m2),
        m2_50x50,
        overlay_mask_on_img(img_arr, m2_50x50),
        orig_m2,
        overlay_mask_on_img(img_arr, orig_m2),
    ]
)